# EarlyStruct – Design Explorer (Floors)
Carbon-first feasibility & comparison tool.

**How to use**
1) Set **Data dir** to the folder containing your CSVs.
2) (Optional) Provide a **Control file** path. If present, it takes precedence for project details and SPANS unless it contains `USE_CSV = Y`.
3) (Optional) Enter **Spans** like `9, 10.5` (meters) or `28ft, 32ft`. Notebook spans override control-file SPANS.
4) Click **Run Evaluation**. If no spans are provided (and no grid_options.csv / ideal spacing), the tool runs a 1-ft sweep (18–45 ft).


In [1]:
import sys
from pathlib import Path

import pandas as pd

CODE_ROOT = Path("/Users/benjaminsalop/Desktop/Oxford/Research/edca/code")

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from earlystruct import cli
from earlystruct.core import reporting


In [2]:
data_dir = Path("/Users/benjaminsalop/Desktop/Oxford/Research/edca/csvs")
control_file = "/Users/benjaminsalop/Desktop/Oxford/Research/control_file/control_file.txt"

df, ranked, pareto, saved = cli.evaluate(
    data_dir=data_dir,
    spans_str=None,          # let control file + sweep logic handle it
    export_dir=None,
    control_file=control_file,
)

if isinstance(saved, dict):
    project = saved.get("project", {}) or {}
    ctl = saved.get("ctl", {}) or {}
else:
    project = {}
    ctl = {}

df.shape, df.columns

((2924, 26),
 Index(['span_input_m', 'span_m', 'span_x_m', 'span_y_m', 'span_slab_dir_m',
        'span_beam_dir_m', 'system_id', 'system_name', 'category', 'type',
        'manufacturer', 'feasible', 'reason', 'depth_m', 'area_m2',
        'concrete_m3', 'steel_m3', 'timber_m3', 'carbon_total_kg',
        'carbon_per_m2', 'cost_total', 'cost_per_m2', 'nx', 'ny',
        'edge_canti_x_m', 'edge_canti_y_m'],
       dtype='object'))

In [3]:
UNIT = ctl.get("UNIT", "metric").lower()
is_imperial = UNIT.startswith("imp") or ("ft" in UNIT)

FT_PER_M = 3.28084

df_disp = df.copy()

if is_imperial:
    df_disp["span_slab_dir_ft"] = df_disp["span_slab_dir_m"] * FT_PER_M
    df_disp["span_beam_dir_ft"] = df_disp["span_beam_dir_m"] * FT_PER_M
else:
    df_disp["span_slab_dir_ft"] = pd.NA
    df_disp["span_beam_dir_ft"] = pd.NA

cols = [
    "system_id", "type", "feasible",
    "span_slab_dir_m", "span_beam_dir_m",
    "span_slab_dir_ft", "span_beam_dir_ft",
    "depth_m", "area_m2",
    "concrete_m3", "steel_m3", "timber_m3",
    "cost_per_m2", "carbon_per_m2",
    "cost_total", "carbon_total_kg",
]

df_disp[[c for c in cols if c in df_disp.columns]].head(15)


,system_id,type,feasible,span_slab_dir_m,span_beam_dir_m,span_slab_dir_ft,span_beam_dir_ft,depth_m,area_m2,concrete_m3,steel_m3,timber_m3,cost_per_m2,carbon_per_m2,cost_total,carbon_total_kg
0,ComFlor100_normal_weight,composite_deck,False,8.0,8.0,<NA>,<NA>,0.170900,20000.0,1923.33334,923.2,NaN,736.252000,769.888792,1.472504e+07,1.539778e+07
1,ComFlor210_normal_weight,composite_deck,False,8.0,8.0,<NA>,<NA>,0.281250,20000.0,3300.83334,1584.4,NaN,19.805000,57.632550,3.961000e+05,1.152651e+06
2,ComFlor80_normal_weight,composite_deck,False,8.0,8.0,<NA>,<NA>,0.140900,20000.0,1923.33334,923.2,NaN,736.252000,769.888792,1.472504e+07,1.539778e+07
3,DuraStress_12DT24_2in_top,double_tee,False,8.0,8.0,<NA>,<NA>,0.059267,20000.0,10344.82758,30000.0,NaN,62.068965,180.620690,1.241379e+06,3.612414e+06
4,DuraStress_12DT24_no_top,double_tee,False,8.0,8.0,<NA>,<NA>,0.055033,20000.0,6896.55172,20000.0,NaN,41.379310,120.413793,8.275862e+05,2.408276e+06
5,DuraStress_12DT26_no_top,double_tee,False,8.0,8.0,<NA>,<NA>,0.059275,20000.0,7310.34482,21200.0,NaN,43.862069,127.638621,8.772414e+05,2.552772e+06
6,DuraStress_12DT28_2in_top,double_tee,False,8.0,8.0,<NA>,<NA>,0.067725,20000.0,11034.48276,32000.0,NaN,66.206897,192.662069,1.324138e+06,3.853241e+06
7,DuraStress_12DT28_no_top,double_tee,False,8.0,8.0,<NA>,<NA>,0.063492,20000.0,7586.20690,22000.0,NaN,45.517241,132.455172,9.103448e+05,2.649103e+06
8,DuraStress_12DT28_pretop,double_tee,False,8.0,8.0,<NA>,<NA>,0.063508,20000.0,10068.96552,29200.0,NaN,60.413793,175.804138,1.208276e+06,3.516083e+06
9,DuraStress_12DT30_pretop,double_tee,False,8.0,8.0,<NA>,<NA>,0.067725,20000.0,10482.75862,30400.0,NaN,62.896552,183.028966,1.257931e+06,3.660579e+06


In [4]:
best_by_type = reporting.cheapest_span_by_type(df)

best_cols = [
    "type",
    "system_id",
    "span_slab_dir_m",
    "span_beam_dir_m",
    "depth_m",
    "area_m2",
    "concrete_m3", "steel_m3", "timber_m3",
    "cost_total", "cost_per_m2",
    "carbon_total_kg", "carbon_per_m2",
]

best_by_type[[c for c in best_cols if c in best_by_type.columns]]


,type,system_id,span_slab_dir_m,span_beam_dir_m,depth_m,area_m2,concrete_m3,steel_m3,timber_m3,cost_total,cost_per_m2,carbon_total_kg,carbon_per_m2
0,double_tee,Wells_DT2010,8.0,8.0,0.093218,20000.0,35416.66666,17000.0,NaN,4.250000e+06,212.500000,1.236750e+07,618.375000
1,hollowcore,bison_hc_200,8.0,8.0,0.200000,20000.0,427.58620,1240.0,NaN,7.696552e+04,3.848276,1.652193e+05,8.260965
2,plank,fpmccann_plank_100_100_y,8.0,8.0,0.200000,20000.0,3958.33334,1900.0,NaN,7.125000e+05,35.625000,1.529500e+06,76.475000
3,solid_plank,bison_sp_200,8.0,8.0,0.200000,20000.0,2583.33334,1240.0,NaN,3.100000e+05,15.500000,9.021000e+05,45.105000


In [5]:
reporting.print_verbose_summary(df, project, ctl)


=== Parametric Floor Design Summary ===
Project: Unnamed project
Unit system: metric

-- Span sweep --
  Using explicit spans from control / CLI

-- One-way irregular slabs --
  Disabled

-- Best option per floor type (sorted by total cost) --

[Type: hollowcore]
  System: bison_hc_200
  Spans (slab × beam): 8.00 m × 8.00 m
  Floor depth: 200 mm
  Building floor area covered: 20000 m²
  Materials: concrete 427.6 m³, steel 1240.0 m³, timber nan m³
  Cost: 76,966 total (3.8 per m²)
  Carbon: 165.2 tCO₂e (8.3 kgCO₂e/m²)

[Type: solid_plank]
  System: bison_sp_200
  Spans (slab × beam): 8.00 m × 8.00 m
  Floor depth: 200 mm
  Building floor area covered: 20000 m²
  Materials: concrete 2583.3 m³, steel 1240.0 m³, timber nan m³
  Cost: 310,000 total (15.5 per m²)
  Carbon: 902.1 tCO₂e (45.1 kgCO₂e/m²)

[Type: plank]
  System: fpmccann_plank_100_100_y
  Spans (slab × beam): 8.00 m × 8.00 m
  Floor depth: 200 mm
  Building floor area covered: 20000 m²
  Materials: concrete 3958.3 m³, steel 190

In [6]:
import sys
from pathlib import Path

# 1) Make sure Python sees your "code" directory as a package root
CODE_ROOT = Path("/Users/benjaminsalop/Desktop/Oxford/Research/edca/code")
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

# 2) Import the parser from the *package* module
from earlystruct.core.control import parse_control

control_path = "/Users/benjaminsalop/Desktop/Oxford/Research/control_file/control_file.txt"

ctl = parse_control(control_path)

print("PROJECT_NAME:", ctl.get("PROJECT_NAME"))
print("UNIT:", ctl.get("UNIT"))
print("FLOOR_AREA_PER_FLOOR:", ctl.get("FLOOR_AREA_PER_FLOOR"))
print("SPAN_SWEEP_FROM_MIN:", ctl.get("SPAN_SWEEP_FROM_MIN"), "BOOL:", ctl.get("SPAN_SWEEP_FROM_MIN_BOOL"))
print("ONE_WAY_IRREGULAR:", ctl.get("ONE_WAY_IRREGULAR"), "BOOL:", ctl.get("ONE_WAY_IRREGULAR_BOOL"))
print("ONE_WAY_SLAB_MIN_SPAN:", ctl.get("ONE_WAY_SLAB_MIN_SPAN"))
print("ONE_WAY_BEAM_MIN_SPAN:", ctl.get("ONE_WAY_BEAM_MIN_SPAN"))
print("PROGRAM_BLOCKS:", ctl.get("PROGRAM_BLOCKS"))


PROJECT_NAME: None
UNIT: None
FLOOR_AREA_PER_FLOOR: None
SPAN_SWEEP_FROM_MIN: None BOOL: None
ONE_WAY_IRREGULAR: None BOOL: None
ONE_WAY_SLAB_MIN_SPAN: None
ONE_WAY_BEAM_MIN_SPAN: None
PROGRAM_BLOCKS: None


In [ ]:
import os
from pathlib import Path
import sys

CODE_ROOT = Path("/Users/benjaminsalop/Desktop/Oxford/Research/edca/code")
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from earlystruct.core.control import parse_control

control_path = "/Users/benjaminsalop/Desktop/Oxford/Research/edca/control_file/control_file.txt"

print("Exists?", os.path.exists(control_path), "->", control_path)

if os.path.exists(control_path):
    with open(control_path, "r", encoding="utf-8", errors="ignore") as f:
        print("\n--- File contents (first 20 lines) ---")
        for i, line in enumerate(f):
            if i >= 20:
                break
            print(repr(line.rstrip("\n")))

ctl = parse_control(control_path)
print("\nParsed keys:", list(ctl.keys()))


Exists? False -> /Users/benjaminsalop/Desktop/Oxford/Research/control_file/control_file.txt

Parsed keys: []
